# Yellow Taxi — Feature Selection & Engineering

Feature-elimination analysis for the Yellow Taxi track: **variance / degeneracy check,
correlation pruning, multicollinearity (VIF), and leakage classification**. Decisions are
logged to `results/yellow_feature_selection.csv`, and written up in
[`docs/yellow_dropped_and_engineered_features.md`](../docs/yellow_dropped_and_engineered_features.md).

**Scope.** This is a *descriptive inference* project (estimate the fee's burden and its
association with volume), not prediction — so "feature selection" means removing
**redundant, degenerate, leaky, or unreliable** variables, not maximizing predictive fit.
The general leakage framework (2024-safe / 2025-forbidden, manufactured correlation) is
project-wide and documented in
[`presentation/feature_leakage.markdown`](../presentation/feature_leakage.markdown); it is
*referenced*, not re-derived, here.

**Data.** The reproducible 20K representative Yellow sample
(`data/processed/samples/yellow_taxi_trip_level_sample_20k_representative.csv`), restricted
to **card/cash** trips (payment_type 1, 2) — the primary burden population. Full-data
validation of these patterns is in `notebooks/yellow_taxi_full_EDA.ipynb` (Part II).

**How to read this.** The candidate features and their roles (treatment / outcome / control) follow the identification strategy in [`modeling_plan.md`](../docs/modeling_plan.md) — this notebook does **not** screen a large feature pool. It **confirms** that the design-selected features are usable: not degenerate (§1), not redundant (§2–§4), and leakage-clean (§5), and records the keep / drop / engineer decisions. Roles are referenced here, not re-derived; the design lives in `modeling_plan.md`.

**Two parts.** **Part A — Selection** (§1–§6): confirm the design-selected features (degeneracy, redundancy, VIF, leakage) and log the keep/drop/engineer decisions to `results/yellow_feature_selection.csv`. **Part B — Engineering** (below): construct and validate the *new* engineered features (`charged_geo`, `base_cost_ex_cbd`). The heavier construction — zone×direction aggregation, DS_z, volume — is done in the **DS_z pipeline** (mirroring HVFHV; there is **no separate `transformers.py`**).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# Resolve repo root robustly (works whether cwd is repo root or notebooks/).
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "data" / "processed").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

SAMPLE = REPO_ROOT / "data" / "processed" / "samples" / "yellow_taxi_trip_level_sample_20k_representative.csv"
RESULTS = REPO_ROOT / "results" / "yellow_feature_selection.csv"
RESULTS.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SAMPLE)
cc = df[df["payment_type"].isin([1, 2])].copy()          # card/cash = primary burden population
cc["trip_duration_minutes"] = cc["trip_duration_seconds"] / 60.0
print(f"All sample rows: {len(df):,} | card/cash rows: {len(cc):,}")
print(cc.groupby('year').size().rename('rows'))
print("card/cash missingness:", int(cc.isna().sum().sum()), "(Flex, missing these fields by design, is excluded)")

All sample rows: 20,000 | card/cash rows: 16,059
year
2024    8115
2025    7944
Name: rows, dtype: int64
card/cash missingness: 0 (Flex, missing these fields by design, is excluded)


## 1. Variance / degeneracy check

A near-constant feature carries no information → drop it (the standard "variance threshold"
step). We run it on the **features we actually intend to use** — kept controls, the cost
outcome, the engineered flag, and the policy/treatment variables — to catch any that is
accidentally degenerate, and specifically any that is **constant in the pre-policy year**
(which rules it out as a cross-year variable). Already-dropped fee components (`extra`,
`mta_tax`, `improvement_surcharge`) are **not** checked here — they are excluded for
double-counting (see the audit), not for low variance, and are not in the standardized schema.

In [2]:
cc["trip_duration_minutes"] = cc["trip_duration_seconds"] / 60
cc["airport_trip_flag"] = (cc["airport_fee"] > 0) & (
    cc["PULocationID"].isin([132, 138]) | cc["DOLocationID"].isin([132, 138]))

candidates = {
    "trip_distance_miles":   "control (kept)",
    "passenger_cost_pretip": "cost / outcome (kept)",
    "pickup_hour":           "time (EDA-only)",
    "day_of_week":           "time (EDA-only)",
    "airport_trip_flag":     "engineered flag (EDA-only)",
    "trip_duration_minutes": "dropped (redundant) — shown for completeness",
    "cbd_congestion_fee":    "policy / treatment",
    "charged_cbd_flag":      "policy / treatment",
}
rows = []
for f, role in candidates.items():
    s = pd.to_numeric(cc[f], errors="coerce")
    by = cc.groupby("year")[f].apply(lambda x: pd.to_numeric(x, errors="coerce").std(ddof=0))
    rows.append({"feature": f, "role": role,
                 "std_overall": s.std(ddof=0), "std_2024": by.get(2024, np.nan),
                 "std_2025": by.get(2025, np.nan),
                 "near_constant_overall": bool(s.nunique() <= 1),
                 "near_constant_2024": bool(by.get(2024, np.nan) == 0)})
var_tbl = pd.DataFrame(rows)
print(var_tbl.round(3).to_string(index=False))
print("\ncharged_cbd_flag mean by year (card/cash):")
print(cc.groupby("year")["charged_cbd_flag"].mean().round(4))

              feature                                         role  std_overall  std_2024  std_2025  near_constant_overall  near_constant_2024
  trip_distance_miles                               control (kept)        4.395     4.403     4.387                  False               False
passenger_cost_pretip                        cost / outcome (kept)       19.642    19.946    19.326                  False               False
          pickup_hour                              time (EDA-only)        5.662     5.726     5.594                  False               False
          day_of_week                              time (EDA-only)        1.934     1.927     1.941                  False               False
    airport_trip_flag                   engineered flag (EDA-only)        0.273     0.268     0.278                  False               False
trip_duration_minutes dropped (redundant) — shown for completeness       26.034    30.199    20.932                  False               False

**Finding.** Every feature we intend to use — `trip_distance_miles`, `passenger_cost_pretip`,
`pickup_hour`, `day_of_week`, `airport_trip_flag` — is **non-degenerate** (healthy variance,
non-zero in both years), so none is dropped for low information. The **only** degeneracy is in
the policy/treatment variables: `cbd_congestion_fee` and `charged_cbd_flag` have **std = 0 in
2024** (fee mean 0 → 0.75 charged share in 2025). They are strictly post-policy → usable as the
*treatment definition*, but **not** as cross-year variables → `charged_cbd_flag` is replaced by a
geographic `charged_geo` (constant across years) for any pre/post design.

## 2. Correlation pruning (trip-economics candidates)

The candidate continuous controls (distance, duration, and the cost/fare family) are
screened for redundancy. We report **Pearson and Spearman** — Pearson is attenuated by the
heavy right tail, so Spearman (rank) is the more honest redundancy signal here.

In [3]:
econ = ["trip_distance_miles", "trip_duration_minutes", "fare_amount", "total_amount",
        "passenger_cost_pretip", "passenger_cost_excl_cbd", "tolls", "airport_fee", "tip_amount"]
sub = cc[econ].apply(pd.to_numeric, errors="coerce")
sub = sub[(sub["trip_distance_miles"] >= 0) & (sub["trip_distance_miles"] < 200)]  # cap extreme distances (>200mi; occur in both regimes)

pear = sub.corr("pearson")
spear = sub.corr("spearman")
print("Pearson r:\n", pear.round(3), "\n")
print("distance vs others  (pearson | spearman):")
for c in ["trip_duration_minutes", "fare_amount", "total_amount", "passenger_cost_pretip"]:
    print(f"  distance ~ {c:24s} {pear.loc['trip_distance_miles', c]:.3f} | {spear.loc['trip_distance_miles', c]:.3f}")
print(f"\ncost family: fare~pretip={pear.loc['fare_amount','passenger_cost_pretip']:.3f}, "
      f"total~pretip={pear.loc['total_amount','passenger_cost_pretip']:.3f}, "
      f"pretip~excl_cbd={pear.loc['passenger_cost_pretip','passenger_cost_excl_cbd']:.3f}")

Pearson r:
                          trip_distance_miles  trip_duration_minutes  fare_amount  total_amount  passenger_cost_pretip  passenger_cost_excl_cbd  tolls  \
trip_distance_miles                    1.000                  0.439        0.903         0.904                  0.911                    0.912  0.667   
trip_duration_minutes                  0.439                  1.000        0.435         0.427                  0.431                    0.432  0.276   
fare_amount                            0.903                  0.435        1.000         0.979                  0.990                    0.991  0.632   
total_amount                           0.904                  0.427        0.979         1.000                  0.992                    0.991  0.713   
passenger_cost_pretip                  0.911                  0.431        0.990         0.992                  1.000                    1.000  0.710   
passenger_cost_excl_cbd                0.912                  0.432   

**Finding.** The **cost/fare family** (`fare_amount`, `total_amount`, `passenger_cost_pretip`,
`passenger_cost_excl_cbd`) is mutually collinear at r ≈ 0.98–1.00 — mechanically the same
quantity. **As controls they are redundant** → keep one, `passenger_cost_pretip`, and drop
`fare_amount` / `total_amount`. Note `passenger_cost_excl_cbd` (= pretip − CBD fee, i.e. the
**base cost**; near-identical to pretip because the fee is $0 pre-policy and only $0.75 on
charged 2025 trips) is **not dropped** — it is redundant only *as a raw control*, but is
**engineered into `base_cost_ex_cbd`, the DS_z denominator**. **Distance** tracks the cost
family (r ≈ 0.90) and **duration** (Spearman ≈ 0.85, Pearson only 0.44 due to outliers) →
duration adds little independent information beyond distance.

## 3. Multicollinearity (VIF)

VIF quantifies how much each candidate is explained by the others. VIF > 10 is the usual
"redundant" threshold.

In [4]:
vifset = ["trip_distance_miles", "trip_duration_minutes", "fare_amount",
          "total_amount", "passenger_cost_pretip"]
X = sub[vifset].dropna().copy()
X["_const"] = 1.0
cols = vifset + ["_const"]
vif = {c: variance_inflation_factor(X[cols].values, i) for i, c in enumerate(cols)}
vif_tbl = pd.DataFrame({"feature": vifset, "VIF": [vif[c] for c in vifset]}).sort_values("VIF", ascending=False)
print(vif_tbl.round(2).to_string(index=False))

              feature    VIF
passenger_cost_pretip 141.12
         total_amount  62.32
          fare_amount  54.74
  trip_distance_miles   5.98
trip_duration_minutes   1.25


**Finding.** The cost family blows up together — `passenger_cost_pretip` (VIF ≈ 141),
`total_amount` (≈ 62), `fare_amount` (≈ 55) — confirming they are one dimension → keep
`passenger_cost_pretip`, drop the others. `trip_distance_miles` (≈ 6) is moderate.
**`trip_duration_minutes` (≈ 1.3) *looks* independent — but that is a Pearson artifact**:
heavy-tailed outliers deflate its linear correlation. In rank terms it is highly collinear with
fare/DS_z (**Spearman(DS_z, duration) = −0.94**, even above distance's −0.89 — see §4), so it is
**not** a cleaner control. **Decision:** keep `passenger_cost_pretip` (cost/denominator) +
`trip_distance_miles` (physical control); drop `fare_amount`, `total_amount`; drop
`trip_duration_minutes` (rank-collinear + endogenous to traffic + off-scope). `tolls` /
`airport_fee` are excluded from this VIF — they are not control candidates.

## 4. Burden-metric redundancy: `relative_cbd_burden`, DS_z, and 1/distance

On 2025 charged card/cash trips (base cost ≥ \$1 floor), we check (a) whether the existing
`relative_cbd_burden` duplicates the project's DS_z metric, and (b) the
structural collinearity **DS_z ≈ 1/base_cost** — a general property of the DS_z metric (not
yellow-specific), for which distance is the weaker proxy.

**Scope:** this is **trip-level, 2025** — the same trips that define DS_z — so it establishes the *definitional* collinearity (why **2025** cost/distance can't be controls; also leakage). It is **not** the Model-1 control question, which uses the **2024 zone-level** baseline (Spearman ≈ −0.88; see the finding + `modeling_plan`).

In [5]:
ch = cc[(cc["year"] == 2025) & (cc["charged_cbd_flag"])].copy()
ch["base_cost"] = (ch["passenger_cost_pretip"] - ch["cbd_congestion_fee"]).round(2)
ch = ch[ch["base_cost"] >= 1.0]
ch["ds_z_trip"] = ch["cbd_congestion_fee"] / ch["base_cost"]          # DS_z definition, trip level
ch["inv_distance"] = 1.0 / ch["trip_distance_miles"].replace(0, np.nan)

print(f"n (2025 charged card/cash, base>=$1): {len(ch):,}")
print(f"corr(relative_cbd_burden, DS_z)      = {ch['relative_cbd_burden'].corr(ch['ds_z_trip']):.4f}")
print(f"corr(DS_z, 1/distance)  Spearman     = {ch['ds_z_trip'].corr(ch['inv_distance'], method='spearman'):.4f}")
print(f"corr(DS_z, distance)    Spearman     = {ch['ds_z_trip'].corr(ch['trip_distance_miles'], method='spearman'):.4f}")
print(f"corr(DS_z, base_cost)   Spearman     = {ch['ds_z_trip'].corr(ch['base_cost'], method='spearman'):.4f}   # definitional: DS_z = fee/base_cost")
print(f"corr(DS_z, 1/base_cost) Pearson      = {ch['ds_z_trip'].corr(1.0/ch['base_cost']):.4f}")
print(f"corr(DS_z, fare_amount) Spearman     = {ch['ds_z_trip'].corr(ch['fare_amount'], method='spearman'):.4f}")
ch["trip_duration_minutes"] = ch["trip_duration_seconds"] / 60
print(f"corr(DS_z, duration)    Spearman     = {ch['ds_z_trip'].corr(ch['trip_duration_minutes'], method='spearman'):.4f}   # > distance -> duration's low VIF is a Pearson artifact")

n (2025 charged card/cash, base>=$1): 5,952
corr(relative_cbd_burden, DS_z)      = 0.9998
corr(DS_z, 1/distance)  Spearman     = 0.9010
corr(DS_z, distance)    Spearman     = -0.8888
corr(DS_z, base_cost)   Spearman     = -1.0000   # definitional: DS_z = fee/base_cost
corr(DS_z, 1/base_cost) Pearson      = 1.0000
corr(DS_z, fare_amount) Spearman     = -0.9877
corr(DS_z, duration)    Spearman     = -0.9404   # > distance -> duration's low VIF is a Pearson artifact


**Finding.** `relative_cbd_burden` and DS_z correlate at **r ≈ 1.00** — near-perfect
duplicates → keep DS_z, demote `relative_cbd_burden` to a reference cross-check.

DS_z is a **deterministic function of *same-year* base cost**: with a near-constant $0.75 fee,
**Spearman(DS_z, base_cost) = −1.00** and **Pearson(DS_z, 1/base_cost) = 1.00** — an algebraic
identity (`fare_amount` Spearman −0.99; distance the weaker proxy, −0.89). So the **2025**
cost/fare/distance of the very trips that define DS_z **cannot** sit beside DS_z in a regression —
they'd control DS_z away, and they're post-policy leakage anyway.

This does **not** forbid Model 1's control, which is the **pre-policy 2024 average distance** — a
*distinct* quantity (different year, not the 2025 base cost). It is only *empirically* correlated
with DS_z (zone-level Spearman ≈ **−0.88**), so it captures the "dense short-trip zone" confound but
is collinear enough that the DS_z coefficient can't be cleanly separated. The DiD designs (M2/M3)
complement this with a control group instead of statistical control.

## 5. Decision table → `results/yellow_feature_selection.csv`

Each candidate feature is logged with its computed screening metrics (where applicable) and
its decision. Leakage status uses the project-wide rule (see the general leakage doc):
`safe_2024` / `forbidden_2025` / `redundant` / `outcome` / `cleaning`.

In [6]:
max_abs = {}
for c in econ:
    off = pear[c].drop(index=c).abs()
    max_abs[c] = off.max()

near_const_2024 = dict(zip(var_tbl["feature"], var_tbl["near_constant_2024"]))

# curated decisions (informed by sections 1-4)
spec = [
 ("pickup_hour","trip","EDA-only","safe_2024","descriptive temporal EDA; NOT a model feature (models are zone-level)"),
 ("day_of_week","trip","EDA-only","safe_2024","descriptive temporal EDA; NOT a model feature (models are zone-level)"),
 ("PULocationID","trip","engineer","safe_2024","-> charged_geo + zone x direction unit; categorical"),
 ("DOLocationID","trip","engineer","safe_2024","-> charged_geo + zone x direction unit; categorical"),
 ("trip_distance_miles","trip","keep","safe_2024","physical control (card/cash); Flex distance weakly-anchored/less reliable (§4), not garbage"),
 ("trip_duration_minutes","trip","drop","redundant","Spearman~0.85 with distance; endogenous; off-scope (speed)"),
 ("passenger_cost_pretip","trip","keep","outcome","cost outcome + DS_z denominator basis (total-tip)"),
 ("cbd_congestion_fee","trip","keep","cleaning","the fee / DS_z numerator; std_2024=0 (post-policy only)"),
 ("charged_cbd_flag","trip","replace","forbidden_2025","std_2024=0; 2025-only -> use charged_geo for cross-year"),
 ("fare_amount","trip","drop","redundant","VIF~135; r~0.99 with pretip"),
 ("total_amount","trip","drop","redundant","VIF~62; = pretip + tip"),
 ("passenger_cost_excl_cbd","trip","engineer","redundant","r~1.00 with pretip; -> base_cost_ex_cbd (denominator)"),
 ("tip_amount","trip","drop","redundant","subtracted out to define pretip; not a covariate"),
 ("relative_cbd_burden","trip","reference","redundant","r~1.00 with DS_z; secondary cross-check only"),
 ("payment_type","trip","engineer","safe_2024","-> regime flags (card/cash vs Flex vs irregular)"),
 ("congestion_surcharge","trip","keep","cleaning","legacy surcharge; base-cost piece; NA on Flex (upfront-priced)"),
 ("tolls","trip","context","cleaning","base-cost piece; not a standalone feature"),
 ("airport_fee","trip","engineer","safe_2024","-> airport_trip_flag (EDA-only, not a model feature; airport analysis deferred); NA on Flex"),
 ("extra","trip","drop","cleaning","bundling -> double-count; not in standardized schema"),
 ("mta_tax","trip","drop","cleaning","base-cost piece; not a feature"),
 ("improvement_surcharge","trip","drop","cleaning","bundling -> double-count; not in standardized schema"),
 ("passenger_count","trip","drop","redundant","low value; NA on Flex (upfront-priced)"),
 ("RatecodeID","trip","drop","redundant","NA on Flex (upfront-priced); marginal use"),
 ("DS_z (mean+median)","zone","primary","outcome","burden metric; 2025 charged card/cash, $1 floor"),
 ("charged_geo","zone","engineer","safe_2024","(PU or DO in CRZ); constant across years -> DiD"),
 ("pct_volume_change","zone","outcome","outcome","non-Flex (card/cash+irregular) n2025/n2024-1; never a predictor"),
 ("n_2024","zone","keep","safe_2024","pre-policy baseline volume; exogenous control"),
 ("avg_distance_2024","zone","keep","safe_2024","pre-policy zone character"),
 ("borough","zone","keep","safe_2024","pre-policy zone character"),
 ("avg_cost_2025 / n_2025 (any 2025 aggregate)","zone","drop","forbidden_2025","post-policy; consequence of shock -> circular"),
]
fs = pd.DataFrame(spec, columns=["feature","level","decision","leakage_status","reason"])
fs["near_constant_2024"] = fs["feature"].map(lambda f: near_const_2024.get(f, ""))
fs["max_abs_corr_pearson"] = fs["feature"].map(lambda f: round(max_abs[f],3) if f in max_abs else "")
fs["VIF"] = fs["feature"].map(lambda f: round(vif[f],2) if f in vif else "")
fs = fs[["feature","level","near_constant_2024","max_abs_corr_pearson","VIF","leakage_status","decision","reason"]]
fs.to_csv(RESULTS, index=False)
print(f"Wrote {RESULTS.relative_to(REPO_ROOT)}  ({len(fs)} features)")
fs

Wrote results/yellow_feature_selection.csv  (30 features)


,feature,level,near_constant_2024,max_abs_corr_pearson,VIF,leakage_status,decision,reason
0,pickup_hour,trip,False,,,safe_2024,EDA-only,descriptive temporal EDA; NOT a model feature ...
1,day_of_week,trip,False,,,safe_2024,EDA-only,descriptive temporal EDA; NOT a model feature ...
2,PULocationID,trip,,,,safe_2024,engineer,-> charged_geo + zone x direction unit; catego...
3,DOLocationID,trip,,,,safe_2024,engineer,-> charged_geo + zone x direction unit; catego...
4,trip_distance_miles,trip,False,0.912,5.98,safe_2024,keep,physical control (card/cash); Flex distance we...
5,trip_duration_minutes,trip,False,0.439,1.25,redundant,drop,Spearman~0.85 with distance; endogenous; off-s...
6,passenger_cost_pretip,trip,False,1.0,141.12,outcome,keep,cost outcome + DS_z denominator basis (total-tip)
7,cbd_congestion_fee,trip,True,,,cleaning,keep,the fee / DS_z numerator; std_2024=0 (post-pol...
8,charged_cbd_flag,trip,True,,,forbidden_2025,replace,std_2024=0; 2025-only -> use charged_geo for c...
9,fare_amount,trip,,0.991,54.74,redundant,drop,VIF~135; r~0.99 with pretip


## 6. Summary

- **Degenerate (post-policy only):** `cbd_congestion_fee`, `charged_cbd_flag` — treatment
  definition, not cross-year variables → use `charged_geo`.
- **Dropped (redundant):** `fare_amount`, `total_amount`, `passenger_cost_excl_cbd` (raw),
  `tip_amount`, `trip_duration_minutes`; `relative_cbd_burden` demoted to reference.
- **Dropped (cleaning/scope):** `extra`, `mta_tax`, `improvement_surcharge`,
  `passenger_count`, `RatecodeID`.
- **Kept controls (2024-safe):** `trip_distance_miles`, `pickup_hour`, `day_of_week`,
  `borough`, `n_2024`, `avg_distance_2024`.
- **Engineered:** `charged_geo`, `base_cost_ex_cbd`, payment-regime flags,
  `airport_trip_flag`, and zone×direction aggregates (`DS_z`, `pct_volume_change`).

The written narrative is in
[`docs/yellow_dropped_and_engineered_features.md`](../docs/yellow_dropped_and_engineered_features.md).

# Part B — Feature Engineering

Construct and validate the *new* engineered features. (The metric-level construction —
zone×direction aggregation, DS_z, `pct_volume_change` — lives in the **DS_z pipeline**,
mirroring HVFHV; there is no separate `transformers.py`.)

*Status: scaffold — to be filled alongside the DS_z pipeline (Phase A/B).*

## B1. `charged_geo` — CRZ membership *(to build)*

Load the **committed CRZ zone lookup** (38 TLC LocationIDs = Manhattan south of 60th St).
Compute `charged_geo = (PULocationID in CRZ) OR (DOLocationID in CRZ)`, for **2024 and 2025**
(unlike `charged_cbd_flag`, which exists only in 2025).

**Validate** against 2025 `charged_cbd_flag`: report agreement (~96.5% expected) and the
disagreement magnitude (charged-in-2025-but-neither-endpoint-in-CRZ = the through-only trips).
If agreement is high, apply the same rule to 2024 → enables the cross-year **charged-vs-control
DiD (M2)**. *(This is where the 96.5% number becomes reproducible.)*

## B2. `base_cost_ex_cbd` — DS_z denominator *(to build)*

`round(passenger_cost_pretip − cbd_congestion_fee, 2)` with a **$1.00 floor**. This is the
denominator of `DS_z = cbd_fee / base_cost_ex_cbd`.

## B3. Already produced upstream (not re-engineered here)

Payment-regime flags (`flex_fare_flag`, `yellow_card_or_cash_flag`, `irregular_payment_flag`)
and `airport_trip_flag` are emitted by [`scripts/standardize_trips.py`](../scripts/standardize_trips.py).